## tl;dr

This notebook compares the completed GEDS crawl with Treasury Board's March 31, 2026 active-employee population. GEDS rows are directory records, not authoritative employee counts.

## Context & Methods

### Key Assumptions

- TBS is the authoritative benchmark for active Pay System employees at March 31, 2026.
- GEDS is a July 8, 2026 directory snapshot with a visible 25-person-per-org ceiling.
- Only explicit normalized-name or reviewed alias matches enter the primary comparison.
- Five non-zero TBS rows without a defensible one-to-one GEDS grain are excluded and retained separately.

## Data

The comparison is stored in `analysis/geds_vs_tbs_2026.sqlite`, with source details and calculations preserved in `analysis/geds_vs_tbs_2026.json`.

In [ ]:
from pathlib import Path
import json
import sqlite3

repo_root = Path.cwd().resolve()
if repo_root.name == "analysis":
    repo_root = repo_root.parent

payload = json.loads((repo_root / "analysis" / "geds_vs_tbs_2026.json").read_text(encoding="utf-8"))
payload["summary"]

## Results

In [ ]:
connection = sqlite3.connect(repo_root / "analysis" / "geds_vs_tbs_2026.sqlite")
size_bands = connection.execute("SELECT * FROM size_bands ORDER BY sort_order").fetchall()
largest_gaps = connection.execute("SELECT tbs_department, tbs_employees_2026, geds_people, directory_ratio FROM comparison ORDER BY (tbs_employees_2026 - geds_people) DESC LIMIT 15").fetchall()
size_bands, largest_gaps

## Takeaways

Use GEDS for organization and person discovery, not employee headcount. The directory ratio declines sharply for the largest departments, and some small organizations exceed 100%, confirming that the two systems use different populations and that GEDS pagination is incomplete.